In [ ]:
include("../src/model.jl")

using SpecialFunctions: erf, erfc
using LinearAlgebra: diag, norm
using Test

using Plots, LaTeXStrings
default(fmt = :png)

In [ ]:
"""
Solve the problem in equilibrium
"""

# Single impurity Anderson model
siam = NCA.SIAM(
  V₀=sqrt(0.15),
  ε₀f=-0.4,
  tilde=true,
  Nc₀=x->NCA.lorentzian(x, 0.0, 0.3)
  )

# Calculate Green functions from the equilibrium NCA solver
nca = NCA.FreqNCA(
  model=siam,
  β=1e2,
  mesh=NCA.LinearMesh(-10.0,10.0,2^15-1)
    )

GᶜA, GᶠA, GᵇA, Aᶜ, Aᵇ_t, Aᶠ_t= begin
  
  imΣ_t = NCA.solve!(nca; project=false)
  
  Ginv = NCA.slave_particle_inv_gf(nca, imΣ_t)
  Gᵇ = 1.0 ./ Ginv[1]
  Gᶠ = 1.0 ./ Ginv[2]
  
  # Trim just the imaginary part
  imGᵇ = view(reinterpret(Float64, Gᵇ), 2:2:length(Gᵇ))
  imGᵇ[imGᵇ .< nca.eps] .= 0.0
  imGᶠ = view(reinterpret(Float64, Gᶠ), 2:2:length(Gᶠ))
  imGᶠ[imGᶠ .< nca.eps] .= 0.0
  
  Gᶜ = NCA.kramerskronig(nca.mesh, π * nca.data.Nc, :real) + 1im * π * nca.data.Nc;
  
  Gᶜ, Gᶠ, Gᵇ, 2.0im * imag(Gᶜ), 1im * 2π * nca.data.Ab_t, 1im * 2π * nca.data.Af_t
end;

# plot(nca.mesh.mesh, GᵇA |> imag, label=L"$Ab$", title="Spectral functions", xlab=L"$\omega$")
# plot!(nca.mesh.mesh, GᶠA |> imag, label=L"$Af$")
# plot!(nca.mesh.mesh, GᶜA |> imag, label=L"$Ac$")

plot(nca.mesh.mesh, Aᵇ_t |> imag, label=L"$Ab_t$", title="Spectral functions", xlab=L"$\omega$")
plot!(nca.mesh.mesh, Aᶠ_t |> imag, label=L"$Af_t$")

## Generating a gaussian pulse

Consider a gaussian pulse
$$\xi(t) = \left(\frac{\Omega^2}{2 \pi}\right)^{1/4}\exp{\left(-\frac{\Omega^2}{4}(t-t_c)^2\right)}$$

Consider a modulator cavity given by the triple $(S_M, L_M, H_M)$

for which $L_M = \lambda(t) a$ where $\lambda(t) \triangleq \frac{\xi(t)}{\sqrt{\int_t^\infty \textrm{d}s |\xi(s)|^2}}$

In [ ]:
model = PhotonAssitedModel(
  #= SIAM parameters =#
  V₀=siam.V₀,
  ε₀f=siam.ε₀f,
  λ₀=siam.λ₀,
  σ=0.3,
  
  #= Light-field parameters =#
  Γ₀=0.00,  # light-matter coupling
  ω₀=1e-1, # photon-ground state energy
  δ₀=1e-1, # photon artificial width
  
  #= Photon-pulse parameters =#
  tc=15,   # peak arrival time
  Ω=1e0,  # frequency bandwitdh of the pulse
  
  #= SLH photon formulation =#
  Γ=5.     # pump and loss strength
)

# Pulse and SLH shape
model.ξ = t -> (model.Ω^2 / (2π))^(1/4)* exp(-(t - model.tc)^2 * (model.Ω^2 / 4))
model.γ = t -> model.Γ * model.ξ(t)
model.λ = t -> model.Γ * model.ξ(t) / sqrt(0.5 * erfc((t - model.tc) * model.Ω / sqrt(2)))
model.hᵧ= t -> (model.ω₀ - 0.5im * (model.λ(t) + model.γ(t)))

# Kadanoff-Baym equations of motion

$$
\left[i \partial_z - h(z) \right]G(z,z^\prime) - \left[\Sigma * G\right](z,z^\prime) = \delta_C(z,z^\prime) \\
\left[-i \partial_z^\prime - h(z^\prime)\right]G(z,z^\prime) - \left[G * \Sigma\right](z,z^\prime)= \delta_C(z,z^\prime)
$$

In [ ]:
data = let n₀ = 0.0 # initial photon-field occupation
  
  # Create a new frequency grid that supports for complex convolutions
  freqs = NCA.LinearMesh(nca.mesh.xmin, nca.mesh.xmax, nca.mesh.L, true);
  
  fermi = NCA.nf.(freqs, nca.β);
  fermi⁻ = NCA.nf.(freqs, -nca.β);
  th½ = tanh.(freqs .* nca.β / 2.0)
  
  # Integral over frequencies
  ∫dω(f) = integrate(freqs.mesh, f, TrapezoidalEvenFast()) / (2π)
  
  GᶜL = ∫dω(fermi .* Aᶜ)
  GᶜL = GreenFunction(GᶜL * ones(1,1), Lesser)
  
  GᶜG = -∫dω(fermi⁻ .* Aᶜ)
  GᶜG = GreenFunction(GᶜG * ones(1,1), Greater)
  
  GᶠL = ∫dω(fermi .* Aᶠ_t)
  GᶠL = GreenFunction(GᶠL * ones(1,1), Lesser)

  GᶠG = -(1.0im)
  GᶠG = GreenFunction(GᶠG * ones(1,1), Greater)

  GᵇL = -∫dω(fermi .* Aᵇ_t)
  GᵇL = GreenFunction(GᵇL * ones(1,1), Lesser)
  
  GᵇG = -(1.0im)
  GᵇG = GreenFunction(GᵇG * ones(1,1), Greater)
  
  GᵞL = -1.0im * n₀
  GᵞL = GreenFunction(GᵞL * ones(1,1), Lesser)
  
  GᵞG = -1.0im * (1 + n₀)
  GᵞG = GreenFunction(GᵞG * ones(1,1), Greater)
  
  Aᶜ˹ = GreenFunction(ElasticArray(hcat(Aᶜ)), MixedLesser)
  
  Aᶠ˹ = GreenFunction(ElasticArray(hcat(Aᶠ_t)), MixedLesser)
  Gᶠ˹p = GreenFunction(ElasticArray(hcat(conj(GᶠA))), MixedLesser)
  
  Aᵇ˹ = GreenFunction(ElasticArray(hcat(Aᵇ_t)), MixedLesser)
  Gᵇ˹p = GreenFunction(ElasticArray(hcat(conj(GᵇA))), MixedLesser)
  
  # Spectral report
  println("Equilibrium spectral function normalizations:")
  println(" ∫dω Aᶜ(ω) = ", ∫dω(Aᶜ) |> imag, ", nᶜ = ", GᶜL[1,1] |> imag)
  println(" ∫dω Aᶠ(ω) = ", ∫dω(fermi⁻ .* Aᶠ_t) |> imag, ", nᶠ = ", GᶠL[1,1] |> imag)
  println(" ∫dω Aᵇ(ω) = ", ∫dω(fermi⁻ .* Aᵇ_t) |> imag, ", nᵇ = ", -GᵇL[1,1] |> imag)
  
  Q = 2GᶠL[1,1] -GᵇL[1,1] |> imag
  println(" Q=$Q")
  
  ProblemData(
    GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,
    Aᶜ˹,Aᶠ˹,Gᶠ˹p,Aᵇ˹,Gᵇ˹p,
    freqs,fermi,fermi⁻,th½);
  end;

calculate_Σ_freq!(model, data, 1);
calculate_Σ_time!(model, data, 1, 1);

In [ ]:
# Initial condition
u0 = tuple(
  [data.GᶜL,data.GᶜG,data.GᶠL,data.GᶠG,data.GᵇL,data.GᵇG,data.GᵞL,data.GᵞG],
  [data.Aᶜ˹,data.Aᶠ˹,data.Gᶠ˹p,data.Aᵇ˹,data.Gᵇ˹p]
)

# Vertical step
f_vert(u, times, t1::Int, t2::Int) = f_vert_time(model, data, times, t1, t2)
f_vert(u, v, times, t1::Int, t2::Int) = f_vert(u, times, t1, t2) .+ f_vert_freq(model, data, times, t1, t2)

# Diagonal step
f_diag(u, times, t1::Int) = (println("t: $(times[t1])"); f_diag_time(model, data, times, t1))
f_diag(u, v, times, t1::Int) = f_diag(u, times, t1) .+ update_norms!(f_diag_freq(model, data, times, t1))

# Vertical step for mixed functions
f_line(u, v, times, t1::Int) = f_line_freq(model, data, times, t1)

thermal_norms = float(Inf)
update_norms!(n) = (global thermal_norms; thermal_norms = norm.(n); n)

"""
Step until the thermal contribution is small and then discard it
"""

# Criteria for evaluating the thermal collision integrals
# TODO: Accumulate norms over horizontal step (don't take into account just the diagonal)
thermal_criteria(norms) = max(norms...) < 1e-8

tspan = (0.0, 100.0)

# The solver will stop when tspan[2] is reached or `stop` evaluates to `false`
state, caches = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2]; f_line=f_line,
    init_dt=1e-3, atol=1e-6, rtol=1e-4, max_dt=0.5,
    stop=()->thermal_criteria(thermal_norms),
    w_line! =(t1)->calculate_Σ_freq!(model,data,t1),
    w_time! =(t1,t2)->calculate_Σ_time!(model,data,t1,t2))

@testset "KMS boundary conditions" begin
  @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Aᶜ˹,Aᶠ˹,Gᶠ˹p,Aᵇ˹,Gᵇ˹p,Σᶠ˹A,Σᵇ˹A,freqs,fermi⁻ = data
  
  imΣ_t = NCA.slave_particle_Σ(nca, nca.data.Ab_t, nca.data.Af_t)
  @test 2 * imΣ_t[:,1] ≈ imag(data.Σᵇ˹A[1])
  @test 2 * imΣ_t[:,2] ≈ imag(data.Σᶠ˹A[1])
  
  # G^R(t,t) = -i
#   @test all(diag(GᶜG.data - GᶜL.data) .≈ -1.0im)
  @test all(diag(GᵞG.data - GᵞL.data) .≈ -1.0im)
  @test all(diag(GᶠG.data) .≈ -1.0im)
  @test all(diag(GᵇG.data) .≈ -1.0im)

  ∫dω(g, f) = (f[abs.(imag(f)) .< 1e-12] .= 0.0; integrate(freqs.mesh, g .* f, TrapezoidalEvenFast()) / (2π))
  ≅(a,b) = isapprox(a,b;atol=1e-3)
  
#   # C^<(t,0) = C˥(t,0+)
  @test all(GᶜL[t,1] ≅ ∫dω(fermi, Aᶜ˹[t]) for t in 1:size(Aᶜ˹,2))
  @test all(GᶠG[1,t] ≅ -∫dω(fermi⁻, Aᶠ˹[t]) for t in 1:size(Aᶠ˹,2))
  @test all(GᵇG[1,t] ≅ -∫dω(fermi⁻, Aᵇ˹[t]) for t in 1:size(Aᵇ˹,2))
  
  @test all(data.ΣᶠG[1,t] ≅ -∫dω(fermi⁻, data.Σᶠ˹A[t]) for t in 1:size(Σᶠ˹A,2))
  @test all(data.ΣᵇG[1,t] ≅ -∫dω(fermi⁻, data.Σᵇ˹A[t]) for t in 1:size(Σᵇ˹A,2))
end

# If we don't need to evaluate the thermal collision integrals anymore
if thermal_criteria(thermal_norms)
  
  # We need a new integrator-state without the mixed functions
  state = KadanoffBaym.VCABMState((state.u[1],), state.t, state.dt)
  
  state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2];
    atol=1e-6, rtol=1e-4, max_dt=1.0,
    w_time! = (t1,t2) -> calculate_Σ_time!(model,data,t1,t2),
    st_ch = (state, caches))
end

# state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2];
#   atol=1e-6, rtol=1e-4, 
#   w_time! =(t1,t2)->calculate_Σ_time!(model,data,t1,t2))

# Extract info
@unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Aᶜ˹,Aᶠ˹,Gᶠ˹p,Aᵇ˹,Gᵇ˹p,Σᶠ˹A,Σᵇ˹A,freqs,fermi = data
times = state.t;

In [ ]:
@unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Aᶜ˹,Aᶠ˹,Gᶠ˹p,Aᵇ˹,Gᵇ˹p,Σᶠ˹A,Σᵇ˹A,freqs,fermi = data
times = state.t;

# Diagonostics
∫dω(g, f) = (f[abs.(imag(f)) .< 1e-12] .= 0.0; integrate(data.freqs.mesh, g .* f, TrapezoidalEvenFast()) / (2π))
∫dω(f) = (f[abs.(imag(f)) .< 1e-12] .= 0.0; integrate(data.freqs.mesh, f, TrapezoidalEvenFast()) / (2π))

# plot(times, imag([GᶜL[1,t] for t in eachindex(times)]))
# plot!(times, imag([∫dω(data.fermi, data.Aᶜ˹[t]) for t in eachindex(times)]))

# Q
plot(times, imag([2GᶠL[t,t] - GᵇL[t,t] for t in eachindex(times)]), xlabel="t", ylabel="Q")

# # Greater tests

# plot(times, imag([GᶠG[1,t] for t in eachindex(times)]))
# plot!(times, imag([-∫dω(data.fermi⁻, data.Aᶠ˹[t]) for t in eachindex(times)]))
# plot(times, real([data.GᵇG[1,t] for t in eachindex(times)]))
# plot!(times, real(-[∫dω(data.fermi⁻, data.Aᵇ˹[t]) for t in eachindex(times)]))

# plot(times, imag([data.ΣᶠG[1,t] for t in eachindex(times)]))
# plot!(times, imag(-[∫dω(data.fermi⁻, data.Σᶠ˹A[t]) for t in eachindex(times)]))
# plot(times, imag([data.ΣᵇG[1,t] for t in eachindex(times)]))
# plot!(times, imag(-[∫dω(data.fermi⁻, data.Σᵇ˹A[t]) for t in eachindex(times)]))

# # Lesser tests (not appropriate because of the real part)
# plot(times, imag([GᶠL[t,1] for t in eachindex(times)]))
# plot!(times, imag([∫dω(data.fermi, data.Aᶠ˹[t]) for t in eachindex(times)]))
# plot(times, imag([-data.GᵇL[t,1] for t in eachindex(times)]))
# plot!(times, imag([∫dω(data.fermi, data.Aᵇ˹[t]) for t in eachindex(times)]))

In [ ]:
plot(times, -imag([GᵇL[t,t] for t in eachindex(times)]))
plot!(times, imag([GᶠL[t,t] for t in eachindex(times)]))
# plot(times, imag([GᵇL[t,t] for t in eachindex(times)]))
# plot(times, imag([GᶠL[t,t] for t in eachindex(times)]))

In [ ]:
# Mixed spectral functions

# plot(nca.mesh.mesh, Gᶜ˥m.data[:,1] - Gᶜ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᶜ˥m.data[:,end] - Gᶜ˥p.data[:,end] |> imag)

# plot(nca.mesh.mesh, Gᵇ˥m.data[:,1] - Gᵇ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᵇ˥m.data[:,end] - Gᵇ˥p.data[:,end] |> imag)

# plot(nca.mesh.mesh, Gᶠ˥m.data[:,1] - Gᶠ˥p.data[:,1] |> imag)
# plot!(nca.mesh.mesh, Gᶠ˥m.data[:,end] - Gᶠ˥p.data[:,end] |> imag)

# heatmap(Gᵇ˥m.data - Gᵇ˥p.data |> imag)
# heatmap(Gᶠ˥m.data - Gᶠ˥p.data |> imag)

In [ ]:
# Pulse
plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser")
plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
plot!(twinx(), times, model.ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot, xlabel=L"t")

In [ ]:
using Interpolations

# G0C = GᶜG.data - G
# GG = GᵞG.data .+ model.Γ₀^2 .* GᵞG.data^2 .* (GᶜG.data .* (1.0im * (GᶠL.data .* GᵇG.data)) + GᶜL.data .* (1.0im * (GᶠG.data .* GᵇL.data)))
# GL = GᵞL.data .+ model.Γ₀^2 .* GᵞL.data^2 .* (GᶜL.data .* (1.0im * (GᶠG.data .* GᵇL.data)) + GᶜG.data .* (1.0im * (GᶠL.data .* GᵇG.data)))

# itp = interpolate((times, times), GG - GL, Gridded(Linear()))
# itp = interpolate((times, times), GᶜG.data - GᶜL.data, Gridded(Linear()))
# itp = interpolate((times, times), GᵞL.data, Gridded(Linear()))
itp = interpolate((times, times), 1.0im * (GᶠG.data .* GᵇL.data - GᶠL.data .* GᵇG.data), Gridded(Linear()))

time_eq = range(times[1], stop=times[end], length=length(times))
ρ, (ωs, ts) = wigner_transform([itp(t1,t2) for t1=time_eq, t2=time_eq], ts=time_eq, fourier=true, Nf=1024)

heatmap(ts, ωs, real(ρ); xlabel=L"$T$", ylabel=L"$\omega$")
ylims!(-5.0, 5.0)

In [ ]:
using Interpolations

# G0C = GᶜG.data - G
# GG = GᵞG.data .+ model.Γ₀^2 .* GᵞG.data^2 .* (GᶜG.data .* (1.0im * (GᶠL.data .* GᵇG.data)) + GᶜL.data .* (1.0im * (GᶠG.data .* GᵇL.data)))
# GL = GᵞL.data .+ model.Γ₀^2 .* GᵞL.data^2 .* (GᶜL.data .* (1.0im * (GᶠG.data .* GᵇL.data)) + GᶜG.data .* (1.0im * (GᶠL.data .* GᵇG.data)))

# itp = interpolate((times, times), GG - GL, Gridded(Linear()))
# itp = interpolate((times, times), GᶜG.data - GᶜL.data, Gridded(Linear()))
itp = interpolate((times, times), GᵞG.data - GᵞL.data, Gridded(Linear()))
# itp = interpolate((times, times), -1.0im * (GᶠL.data .* GᵇG.data), Gridded(Linear()))

time_eq = range(times[1], stop=times[end], length=length(times))
ρ, (ωs, ts) = wigner_transform([itp(t1,t2) for t1=time_eq, t2=time_eq], ts=time_eq, fourier=true, Nf=1024)

heatmap(ts, ωs, real(ρ); xlabel=L"$T$", ylabel=L"$\omega$")
# ylims!(-5.0, 5.0)

In [ ]:
using Interpolations

time_eq = range(times[1], stop=times[end], length=length(times))
itp = interpolate((times, times), , Gridded(Linear()))

ρ, (ωs, ts) = wigner_transform([itp(t1,t2) for t1=time_eq, t2=time_eq], ts=time_eq, fourier=true)

heatmap(ts, ωs, real(ρ); xlabel=L"$T$", ylabel=L"$\omega$")
ylims!(-5.0, 5.0)

In [ ]:
# Time-dependent spectral functions

heatmap(times, times, (GᶜG.data - GᶜL.data) |> imag)

time_eq = range(times[1], stop=times[end], length=length(times)÷3)
itp = interpolate((times, times), 1.0im * GᶠG.data .* GᵇL.data - GᶠL.data .* GᵇG.data, Gridded(Linear()))


# ρ, (ωs, ts) = wigner_transform([itp(t1,t2) for t1=time_eq, t2=time_eq], ts=time_eq, fourier=true)

# ρ, (ωs, ts) = wigner_transform(1.0im * (GᶜG.data - GᶜL.data), ts=times)#; Nf=4096*2);
# ρ, (ωs, ts) = wigner_transform(1.0im * (GᶠG.data - GᶠL.data), ts=times)#; Nf=4096*2);
# ρ, (ωs, ts) = wigner_transform(1.0im * (GᵇG.data - GᵇL.data), ts=times)#; Nf=4096*2);
# ρ, (ωs, ts) = wigner_transform(1.0im * (GᵞG.data - GᵞL.data), ts=times)#; Nf=4096*2);

In [ ]:
# plot(times, 1.0im * diag(GᵇL.data - 2GᶠL.data) .- 1.0im * diag(GᵇL.data - 2GᶠL.data)[1] |> real, label="dQ")

# begin
#   @unpack GᶜL,GᶜG,G L,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,freqs = data;
#   times = state.t;
#   ∫dω(f) = integrate(freqs.mesh, f, TrapezoidalEvenFast()) / (2π)
#   plot(times, [∫dω(data.fermi .* (Gᶠ˥m[t] - Gᶠ˥p[t])) for t in eachindex(times)] .|> imag)
#   plot!(times, [GᶠL[t,1] for t in 1:size(Gᶜ˥m,2)] .|> imag)
# end

# begin
#   @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,freqs = data;
#   times = state.t;
#   ∫dω(f) = integrate(freqs.mesh, f, TrapezoidalEvenFast()) / (2π)
#   plot(times, [∫dω(data.fermi .* (Gᶠ˥m[t] - Gᶠ˥p[t])) for t in eachindex(times)] .|> imag)
#   plot!(times, [GᶠL[t,1] for t in 1:size(Gᶜ˥m,2)] .|> imag)
# end

# begin
#   @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Gᶜ˥m,Gᶜ˥p,Gᶠ˥m,Gᶠ˥p,Gᵇ˥m,Gᵇ˥p,freqs = data;
#   times = state.t;
#   ∫dω(f) = integrate(freqs.mesh, f, TrapezoidalEvenFast()) / (2π)
#   plot(times, [∫dω(data.fermi .* (Gᶠ˥m[t] - Gᶠ˥p[t])) for t in 1:size(Gᶜ˥m,2)] .|> imag)
#   plot!(times, [GᶠL[t,1] for t in 1:size(Gᶜ˥m,2)] .|> imag)
# end

# @testset begin
#   @test minimum(imag(GᶠL.data)) > 0.0  # Maximum occupation is 1
#   @test maximum(imag(GᶠL.data)) < 1.0  # Maximum occupation is 1
#   @test minimum(-imag(GᵇL.data)) > 0.0
#   @test maximum(-imag(GᵇL.data)) < 1.0
#   @test minimum(-imag(GᶠG.data)) > 0.0
#   @test maximum(-imag(GᶠG.data)) < 1.0
# end

@show extrema(imag(diag(GᶠG.data)))
@show extrema(imag(diag(GᵇG.data)))
@show extrema(imag(diag(GᵇL.data)))
@show extrema(imag(diag(GᶠL.data)))
@show extrema(imag(GᶠG.data))
@show extrema(imag(GᵇG.data))
@show extrema(imag(GᶠL.data))
@show extrema(imag(GᵇL.data))

In [ ]:
# function refine_grid_2d(f, minmax::Tuple{Number, Number}; max_points = 1000, max_error=1e10)
#   if minmax[1] > minmax[2]
#     throw(ArgumentError("interval must be given as (min, max)"))
#   elseif minmax[1] == minmax[2]
#     x = minmax[1]
#     return [x], [f(x)]
#   end
  
#   # Initial number of points
#   n_points = 21
#   @assert isodd(n_points)
  
#   xs = collect(range(minmax[1]; stop=minmax[2], length=n_points))
  
#   # Move the first and last interior points a bit closer to the end points
# #   xs[2] = xs[1] + (xs[2] - xs[1]) * 0.25
# #   xs[end-1] = xs[end] - (xs[end] - xs[end-1]) * 0.25
  
#   while n_points < max_points
#     fs = [f(x,y) for x in xs, y in xs]

#     mid_idxs = 2:2:n_points

#     err = zeros(eltype(fs), length(mid_idxs), length(mid_idxs))
    
#     for (k1,i1) in enumerate(mid_idxs)
#       for k2 in 1:length(mid_idxs)
#         i2 = mid_idxs[k2]
#         err[k1,k2] = abs(fs[i1,i2] - (fs[i1+1,i2] + fs[i1-1,i2] + fs[i1,i2+1] + fs[i1,i2-1]) / 4)
#       end
#     end
#     println(maximum(err))
#     # In our case the underlying grid has to be the cartesian product of some grid x
#     # So we take the maximum value of the error of each column as the error everywhere
#     idx_to_refine = findall(e -> e > max_error, map(maximum, eachcol(err)))
# #     println(idx_to_refine)
#     if isempty(idx_to_refine)
#       break
#     end
    
#     # Map these to global indices
#     idx_to_refine = 2 * idx_to_refine
    
#     new_xs = similar(xs, length(xs) + 2*length(idx_to_refine))
#     k = 0
#     for i in 1:length(xs)
#       if i in idx_to_refine
#         k += 1
#         new_xs[i-1 + k] = (xs[i] + xs[i-1]) / 2
#         new_xs[i + k]   = xs[i]
#         new_xs[i+1 + k] = (xs[i+1] + xs[i]) / 2
#         k += 1
#       else
#         new_xs[i + k]   = xs[i]
#       end
#     end

#     xs = new_xs
#     n_points = length(xs)
#   end
#   return xs
# end

# using Interpolations

# itp = interpolate((times,times), imag(GᶠG))

In [ ]:
# plot(times, [GᵇL[t,t] for t in 1:size(GᶜL,1)] |> imag)
# plot(times, [GᶠL[t,t] for t in 1:size(GᶜL,1)] |> imag)
# plot(data.freqs.mesh, data.Σᶠ˥m[1] |> real)
# plot!(data.freqs.mesh, NCA.kramerskronig(nca.mesh, -data.Σᶠ˥m[1] |> imag, :real))

# plot(data.freqs.mesh, data.Σᵇ˥m[1] |> real)
# plot!(data.freqs.mesh, NCA.kramerskronig(nca.mesh, data.Σᵇ˥m[1] |> imag, :real))

In [ ]:
# ∫dω(f) = integrate(data.freqs.mesh, f, TrapezoidalEvenFast()) / (2π)

# plot(times, [2GᶠL[t,t] - GᵇL[t,t] for t in 1:size(GᶜL,1)] |> imag)
# plot!(times, [∫dω(data.fermi .* (Gᶠ˥m[t] - Gᶠ˥p[t])) for t in 1:size(Gᶜ˥m,2)] |> imag)

# plot([sum(Gᶜ˥m[t] - Gᶜ˥p[t]) for t in 1:size(Gᶜ˥m,2)] |> imag)

# plot(Gᶜ˥m[1] |> imag)